# Step 3 — SPECTER2 embeddings + Ridge ordinal + threshold tuning

**Goal:** push QWK from the 0.63 TF-IDF anchor toward 0.70 by replacing TF-IDF with SPECTER2 dense embeddings on `title + abstract`.

**Runtime:** designed for Colab A100. The full pipeline is roughly 1-2 min on A100, ~5 min on T4.

## Pipeline
1. Upload data zip with the 5 CSVs.
2. Load + merge title with abstract.
3. Load SPECTER2 base + the `proximity` adapter.
4. Encode all rows -> 768-dim `[CLS]` embeddings, cached as `.npy`.
5. Repeated stratified 5x5 CV Ridge regression on label as float, plus a venue/year/first-author/has-abstract feature stack.
6. Tune 4 thresholds on OOF scores to maximise QWK.
7. Save OOF + test scores + submission CSV.

Download the whole `outputs/specter2_ridge/` folder back to your laptop and drop it under `outputs/specter2_ridge/` in the local repo before running step 5.

## 1. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q --upgrade "transformers>=4.41" "adapters>=0.2.0" "accelerate>=0.30" scikit-learn pandas numpy scipy

## 2. Upload data

Zip the 5 files locally:
```
asp_data.zip
  train.csv
  public_test.csv
  private_test.csv
  Test_Submission.csv
  abstracts_merged_v2.csv
```
Then run the cell below and pick `asp_data.zip`.

In [ ]:
import pathlib, zipfile
from google.colab import files

WORK = pathlib.Path('/content/work')
DATA = WORK / 'data'
OUT = WORK / 'outputs'
EMB_DIR = OUT / 'embeddings'
RUN_DIR = OUT / 'specter2_ridge'
for d in [WORK, DATA, OUT, EMB_DIR, RUN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name, content in uploaded.items():
    target = WORK / name
    target.write_bytes(content)
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(target) as zf:
            zf.extractall(DATA)
        print('Extracted', name, '->', DATA)

for p in sorted(DATA.glob('*')):
    print(p.name, p.stat().st_size)

## 3. Load + merge

In [ ]:
import pandas as pd, numpy as np, json, re
from pathlib import Path

DATA = Path('/content/work/data')
OUT = Path('/content/work/outputs')
EMB_DIR = OUT / 'embeddings'
RUN_DIR = OUT / 'specter2_ridge'

train = pd.read_csv(DATA / 'train.csv')
public = pd.read_csv(DATA / 'public_test.csv')
private = pd.read_csv(DATA / 'private_test.csv')
sample = pd.read_csv(DATA / 'Test_Submission.csv')
abstracts = pd.read_csv(DATA / 'abstracts_merged_v2.csv')

print('train', train.shape, 'public', public.shape, 'private', private.shape)
print('abstracts', abstracts.shape, 'has_abstract', int(abstracts['has_abstract'].sum()))

abs_map = abstracts[['source_split', 'id', 'abstract', 'has_abstract', 'abstract_source']]

def attach_abstract(df, split):
    df = df.copy()
    df['source_split'] = split
    merged = df.merge(abs_map, on=['source_split', 'id'], how='left')
    merged['abstract'] = merged['abstract'].fillna('')
    merged['has_abstract'] = merged['has_abstract'].fillna(False).astype(bool)
    merged['abstract_source'] = merged['abstract_source'].fillna('')
    return merged

train_full = attach_abstract(train, 'train')
public_full = attach_abstract(public, 'public_test')
private_full = attach_abstract(private, 'private_test')

for name, df in [('train', train_full), ('public', public_full), ('private', private_full)]:
    print(f'{name}: rows={len(df)}, has_abstract={int(df["has_abstract"].sum())} ({df["has_abstract"].mean():.1%})')

## 4. Load SPECTER2 + proximity adapter

In [ ]:
import torch
from transformers import AutoTokenizer
from adapters import AutoAdapterModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

tokenizer = AutoTokenizer.from_pretrained('allenai/specter2_base')
model = AutoAdapterModel.from_pretrained('allenai/specter2_base')
model.load_adapter('allenai/specter2', source='hf', load_as='proximity', set_active=True)
model.to(device).eval()
print('SPECTER2 + proximity adapter loaded. Hidden size =', model.config.hidden_size)

## 5. Encode title + abstract -> 768-dim embeddings (cached as .npy)

In [ ]:
from tqdm.auto import tqdm

MAX_LEN = 512
BATCH = 32
SEP = tokenizer.sep_token

def build_input(title, abstract):
    title = '' if pd.isna(title) else str(title).strip()
    abstract = '' if pd.isna(abstract) else str(abstract).strip()
    return f'{title}{SEP}{abstract}' if abstract else title

@torch.no_grad()
def encode(df, name):
    cache = EMB_DIR / f'specter2_{name}.npy'
    if cache.exists():
        emb = np.load(cache)
        if emb.shape[0] == len(df):
            print(f'{name}: cached embeddings shape={emb.shape}')
            return emb
        print(f'{name}: cache size mismatch, re-encoding')
    texts = [build_input(t, a) for t, a in zip(df['title'], df['abstract'])]
    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float32)
    for start in tqdm(range(0, len(texts), BATCH), desc=f'encoding {name}'):
        batch_texts = texts[start:start + BATCH]
        inputs = tokenizer(batch_texts, padding=True, truncation=True,
                           max_length=MAX_LEN, return_tensors='pt',
                           return_token_type_ids=False)
        inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}
        outputs = model(**inputs)
        cls = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        out[start:start + len(batch_texts)] = cls
    np.save(cache, out)
    print(f'{name}: encoded {out.shape} -> {cache}')
    return out

emb_train = encode(train_full, 'train')
emb_public = encode(public_full, 'public')
emb_private = encode(private_full, 'private')
print('shapes:', emb_train.shape, emb_public.shape, emb_private.shape)

## 6. Side features (venue/year one-hot, has_abstract, title length, author count, first surname target encoding)

We concatenate these to the SPECTER2 vector so the Ridge head learns a tiny
metadata residual on top of pure semantic embeddings.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

LABEL = 'Label'

def first_surname(value):
    if pd.isna(value) or not str(value).strip():
        return ''
    first_author = str(value).split(',')[0]
    tokens = re.findall(r"[A-Za-z\u00C0-\u1EF9'\u2019-]+", first_author)
    return tokens[-1].lower() if tokens else ''

def add_basic(df):
    df = df.copy()
    df['first_surname'] = df['authors'].map(first_surname)
    df['has_authors'] = df['authors'].notna().astype(int)
    df['title_len'] = df['title'].fillna('').astype(str).str.len()
    df['author_count'] = df['authors'].fillna('').map(
        lambda v: 0 if not str(v).strip() else len(str(v).split(',')))
    df['abstract_len_norm'] = df['abstract'].fillna('').str.len() / 1000.0
    df['has_abstract_int'] = df['has_abstract'].astype(int)
    return df

train_full = add_basic(train_full)
public_full = add_basic(public_full)
private_full = add_basic(private_full)

venues = sorted(set(train_full['venue']) | set(public_full['venue']) | set(private_full['venue']))
years = sorted(set(train_full['year']) | set(public_full['year']) | set(private_full['year']))
print('venues =', venues, 'years range =', min(years), max(years))

def numeric_block(df):
    venue_oh = pd.get_dummies(df['venue'].astype(str), prefix='venue').reindex(
        columns=[f'venue_{v}' for v in venues], fill_value=0).astype(np.float32).to_numpy()
    yr = pd.to_numeric(df['year'], errors='coerce').fillna(np.nanmedian(years)).astype(float)
    yr_norm = ((yr - 2016.0) / 10.0).to_numpy(dtype=np.float32).reshape(-1, 1)
    misc = df[['has_authors', 'title_len', 'author_count', 'abstract_len_norm', 'has_abstract_int']].astype(np.float32).to_numpy()
    misc[:, 1] = misc[:, 1] / 100.0  # rough scale title length
    misc[:, 2] = misc[:, 2] / 10.0   # rough scale author count
    return np.concatenate([venue_oh, yr_norm, misc], axis=1).astype(np.float32)

side_train = numeric_block(train_full)
side_public = numeric_block(public_full)
side_private = numeric_block(private_full)
print('side dims:', side_train.shape, side_public.shape, side_private.shape)

## 7. Repeated 5x5 CV Ridge regression on SPECTER2 + side features

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import cohen_kappa_score, mean_absolute_error

FOLDS = 5
SEEDS = [252, 253, 254, 255, 256]
ALPHAS = [4.0, 8.0, 16.0]  # we will pick best by OOF QWK

def stack_features(emb, side, l2_normalize=True):
    if l2_normalize:
        norms = np.linalg.norm(emb, axis=1, keepdims=True)
        emb = emb / np.clip(norms, 1e-9, None)
    return np.concatenate([emb.astype(np.float32), side.astype(np.float32)], axis=1)

X_train = stack_features(emb_train, side_train)
X_public = stack_features(emb_public, side_public)
X_private = stack_features(emb_private, side_private)
y = train_full[LABEL].astype(float).to_numpy()
y_class = train_full[LABEL].astype(int).to_numpy()
print('X_train', X_train.shape, 'y dist:', dict(pd.Series(y_class).value_counts().sort_index()))

In [ ]:
def run_repeated_cv(alpha):
    oof_sum = np.zeros(len(X_train), dtype=np.float64)
    oof_count = np.zeros(len(X_train), dtype=np.float64)
    public_sum = np.zeros(len(X_public), dtype=np.float64)
    private_sum = np.zeros(len(X_private), dtype=np.float64)
    n_models = 0
    for seed in SEEDS:
        cv = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=seed)
        for fold_idx, (tr, va) in enumerate(cv.split(X_train, y_class), start=1):
            model_ = Ridge(alpha=alpha, solver='lsqr')
            model_.fit(X_train[tr], y[tr])
            oof_sum[va] += np.clip(model_.predict(X_train[va]), 1.0, 5.0)
            oof_count[va] += 1.0
            public_sum += np.clip(model_.predict(X_public), 1.0, 5.0)
            private_sum += np.clip(model_.predict(X_private), 1.0, 5.0)
            n_models += 1
    return oof_sum / oof_count, public_sum / n_models, private_sum / n_models

results = {}
for alpha in ALPHAS:
    oof, pub, priv = run_repeated_cv(alpha)
    qwk_round = cohen_kappa_score(y_class, np.clip(np.round(oof), 1, 5).astype(int), weights='quadratic')
    print(f'alpha={alpha}: round-QWK={qwk_round:.4f}, oof mean={oof.mean():.3f}')
    results[alpha] = (oof, pub, priv, qwk_round)

## 8. Tune 4 thresholds per alpha to maximise QWK on OOF

In [ ]:
from scipy.optimize import differential_evolution

def scores_to_labels(scores, thresholds):
    return np.digitize(scores, np.sort(np.asarray(thresholds, dtype=float))) + 1

def tune_thresholds(y_true, oof_scores, seed=42):
    def objective(raw):
        thr = np.sort(raw)
        gap = np.min(np.diff(thr))
        penalty = 0.0 if gap >= 0.03 else (0.03 - gap) * 5.0
        return -cohen_kappa_score(y_true, scores_to_labels(oof_scores, thr), weights='quadratic') + penalty
    bounds = [(1.4, 2.5), (1.8, 2.9), (2.2, 3.4), (2.6, 4.2)]
    res = differential_evolution(objective, bounds, seed=seed, maxiter=120, popsize=15,
                                 polish=True, updating='immediate', workers=1)
    thr = np.sort(res.x)
    qwk = cohen_kappa_score(y_true, scores_to_labels(oof_scores, thr), weights='quadratic')
    return thr, qwk

best = None
for alpha, (oof, pub, priv, qwk_round) in results.items():
    thr, qwk = tune_thresholds(y_class, oof, seed=42)
    print(f'alpha={alpha}: tuned QWK={qwk:.4f}  thresholds={thr}')
    if best is None or qwk > best['qwk']:
        best = {'alpha': alpha, 'qwk': qwk, 'thresholds': thr,
                'oof': oof, 'public': pub, 'private': priv}
print()
print('Best alpha =', best['alpha'], 'OOF QWK =', round(best['qwk'], 4))

## 9. Save artefacts + build submission

In [ ]:
from sklearn.metrics import f1_score

oof_pred = scores_to_labels(best['oof'], best['thresholds'])
public_pred = scores_to_labels(best['public'], best['thresholds'])
private_pred = scores_to_labels(best['private'], best['thresholds'])

metrics = {
    'method': 'specter2_ridge',
    'alpha': best['alpha'],
    'oof_qwk': float(best['qwk']),
    'oof_mae': float(mean_absolute_error(y_class, oof_pred)),
    'oof_macro_f1': float(f1_score(y_class, oof_pred, average='macro')),
    'thresholds': [float(v) for v in best['thresholds']],
    'label_distribution_combined': {int(k): int(v) for k, v in pd.Series(
        np.concatenate([public_pred, private_pred])).value_counts().sort_index().items()},
    'label_distribution_public': {int(k): int(v) for k, v in pd.Series(public_pred).value_counts().sort_index().items()},
    'label_distribution_private': {int(k): int(v) for k, v in pd.Series(private_pred).value_counts().sort_index().items()},
}
(RUN_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

pd.DataFrame({'id': train_full['id'], 'Label': y_class,
              'oof_score': best['oof'], 'oof_pred': oof_pred}).to_csv(
    RUN_DIR / 'oof_scores.csv', index=False)
pd.DataFrame({'id': public_full['id'], 'score': best['public'],
              'pred': public_pred}).to_csv(RUN_DIR / 'public_scores.csv', index=False)
pd.DataFrame({'id': private_full['id'], 'score': best['private'],
              'pred': private_pred}).to_csv(RUN_DIR / 'private_scores.csv', index=False)

# Combined submission ordered like Test_Submission.csv
combo = pd.concat([
    pd.DataFrame({'id': public_full['id'], 'Label': public_pred}),
    pd.DataFrame({'id': private_full['id'], 'Label': private_pred}),
], ignore_index=True)
submission = sample[['id']].merge(combo, on='id', how='left')
assert submission['Label'].notna().all() and submission.shape[0] == len(sample)
submission['Label'] = submission['Label'].astype(int)
submission.to_csv(RUN_DIR / 'specter2_ridge_submission.csv', index=False)
print('submission rows =', len(submission), submission.head())

## 10. Zip outputs + download

Download the produced zip, extract into the local repo at `outputs/specter2_ridge/` so step 5 (stacking) can read it.

In [ ]:
import shutil
zip_path = pathlib.Path('/content/specter2_ridge_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in RUN_DIR.iterdir():
        zf.write(p, arcname=f'specter2_ridge/{p.name}')
    for p in EMB_DIR.iterdir():
        zf.write(p, arcname=f'embeddings/{p.name}')
print('zipped:', zip_path, 'size MB =', round(zip_path.stat().st_size / 1e6, 2))
files.download(str(zip_path))